In [1]:
import os

from google.colab import drive
drive.mount('/content/drive')
base_path = "/content/drive/MyDrive/Colab Notebooks/Quant"
os.chdir(base_path)

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.callbacks import LambdaCallback, EarlyStopping, ModelCheckpoint

import tensorflow as tf

from tensorflow import keras
import keras_hub
from tensorflow.keras import regularizers, layers
from tensorflow.keras.layers import Input, Dropout, Dense, Layer, Embedding, Lambda
from keras_hub.layers import PositionEmbedding
from tensorflow.keras.layers import Embedding, MultiHeadAttention, LayerNormalization, GlobalMaxPooling1D, GlobalAveragePooling1D, TextVectorization, BatchNormalization
from tensorflow.keras.models import Model, Sequential

import yfinance as yf
from config import config

from dataclasses import dataclass
import glob
from pprint import pprint

Mounted at /content/drive


In [5]:
def custom_standardization(input_data):
    lowercase = tf.strings.lower(input_data)
    stripped_html = tf.strings.regex_replace(lowercase, "<br />", " ")
    return tf.strings.regex_replace(
        stripped_html, "[%s]" % re.escape("!#$%&'()*+,-./:;<=>?@\\^_`{|}~"), ""
    )

def get_vectorize_layer(texts, vocab_size, max_seq, special_tokens=["[MASK]"]):
    vectorize_layer = TextVectorization(
        max_tokens = vocab_size,
        output_mode = "int",
        standardize = custom_standardization,
        output_sequence_length = max_seq
    )
    vectorize_layer.adapt(texts)

    vocab = vectorize_layer.get_vocabulary()
    vocab = vocab[2: vocab_size - len(special_tokens)] + ["[mask]"]
    vectorize_layer.set_vocabulary(vocab)
    return vectorize_layer

def encode(texts):
    encoded_texts = vectorize_layer(texts)
    return encoded_texts.numpy()

def get_masked_input_and_labels(encoded_texts):
    inp_mask = np.random.rand(*encoded_texts.shape) < 0.15
    inp_mask[encoded_texts <= 2] = False
    labels = -1 * np.ones(encoded_texts.shape, dtype = int)
    labels[inp_mask] = encoded_texts[inp_mask]

    encoded_texts_masked = np.copy(encoded_texts)
    inp_mask_2mask = inp_mask & (np.random.rand(*encoded_texts.shape) < 0.90)
    encoded_texts_masked[inp_mask_2mask] = (mask_token_id)

    inp_mask_2random = inp_mask_2mask & (np.random.rand(*encoded_texts.shape) < 1/9)
    encoded_texts_masked[inp_mask_2random] = np.random.randint(3, mask_token_id, inp_mask_2random.sum())

    sample_weights = np.ones(labels.shape)
    sample_weights[labels == -1] = 0

    y_labels = np.copy(encoded_texts)

    return encoded_texts_masked, y_labels, sample_weights

In [8]:
# Dataset for MLM Pretraining For Google Colab:
news_raw = pd.read_csv(os.path.join("data", "abcnews-date-text.csv"))
news_text_raw = news_raw["headline_text"]

vectorize_layer = get_vectorize_layer(
    news_text_raw.tolist(),
    config.VOCAB_SIZE,
    config.MAX_LEN,
    special_tokens=["[mask]"],
)
mask_token_id = vectorize_layer(["[mask]"]).numpy()[0][0]

# mlm_ds was created on a local machine, then added to the drive
mlm_ds = tf.data.Dataset.load(os.path.join("data", "mlm_dataset_final"))
mlm_ds_small = mlm_ds.shard(num_shards=12, index=0)

# Dataset For Sentiment Classification
sentiment_raw = pd.read_csv(os.path.join("data", "sentiment.csv"), encoding='latin1', header = None)
sentiment_raw.columns = ["Output", "Input"]
sentiment_dataset = sentiment_raw[["Input", "Output"]]

df_0 = sentiment_dataset[sentiment_dataset["Output"] == "negative"]
df_1 = sentiment_dataset[sentiment_dataset["Output"] == "neutral"]
df_2 = sentiment_dataset[sentiment_dataset["Output"] == "positive"]

min_label = min(len(df_0), len(df_1), len(df_2))
df_0_downsampled = resample(df_0, replace=False, n_samples=min_label, random_state=42)
df_1_downsampled = resample(df_1, replace=False, n_samples=min_label, random_state=42)
df_2_downsampled = resample(df_2, replace=False, n_samples=min_label, random_state=42)

sentiment_dataset_balanced = pd.concat([df_0_downsampled, df_1_downsampled, df_2_downsampled])

X, y = sentiment_dataset_balanced['Input'], sentiment_dataset_balanced['Output']
X_encoded = encode(X)

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.1, random_state=42
)

train_classifier_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(1000).batch(config.BATCH_SIZE))
test_classifier_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(config.BATCH_SIZE)

In [9]:
# # Dataset for MLM Pretraining For Local Machine
# news_raw = pd.read_csv(os.path.join("data", "abcnews-date-text.csv"))
# news_text_raw = news_raw["headline_text"]

# vectorize_layer = get_vectorize_layer(
#     news_text_raw.tolist(),
#     config.VOCAB_SIZE,
#     config.MAX_LEN,
#     special_tokens=["[mask]"],
# )

# mask_token_id = vectorize_layer(["[mask]"]).numpy()[0][0]

# x_all_encoded = encode(news_text_raw)

# x_masked_train, y_masked_labels, sample_weights = get_masked_input_and_labels(x_all_encoded)

# mlm_ds = tf.data.Dataset.from_tensor_slices(
#     (x_masked_train, y_masked_labels, sample_weights)
# )
# mlm_ds = mlm_ds.shuffle(1000).batch(config.BATCH_SIZE)
# mlm_ds_small = mlm_ds.shard(num_shards=256, index=0)

# # Dataset For Sentiment Classification
# sentiment_raw = pd.read_csv(os.path.join("data", "sentiment.csv"), encoding='latin1', header = None)
# sentiment_raw.columns = ["Output", "Input"]
# sentiment_dataset = sentiment_raw[["Input", "Output"]]

# df_0 = sentiment_dataset[sentiment_dataset["Output"] == "negative"]
# df_1 = sentiment_dataset[sentiment_dataset["Output"] == "neutral"]
# df_2 = sentiment_dataset[sentiment_dataset["Output"] == "positive"]

# min_label = min(len(df_0), len(df_1), len(df_2))
# df_0_downsampled = resample(df_0, replace=False, n_samples=min_label, random_state=42)
# df_1_downsampled = resample(df_1, replace=False, n_samples=min_label, random_state=42)
# df_2_downsampled = resample(df_2, replace=False, n_samples=min_label, random_state=42)

# sentiment_dataset_balanced = pd.concat([df_0_downsampled, df_1_downsampled, df_2_downsampled])

# X, y = sentiment_dataset_balanced['Input'], sentiment_dataset_balanced['Output']
# X_encoded = encode(X)

# le = LabelEncoder()
# y_encoded = le.fit_transform(y)

# X_train, X_test, y_train, y_test = train_test_split(
#     X_encoded, y_encoded, test_size=0.1, random_state=42
# )

# train_classifier_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(1000).batch(config.BATCH_SIZE))
# test_classifier_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(config.BATCH_SIZE)

In [ ]:
def bert_module(query, key, value, i):
    # Multi headed self-attention
    attention_output = layers.MultiHeadAttention(
        num_heads=config.NUM_HEAD,
        key_dim=config.EMBED_DIM // config.NUM_HEAD,
        name="encoder_{}_multiheadattention".format(i),
    )(query, key, value)
    attention_output = layers.Dropout(0.1, name="encoder_{}_att_dropout".format(i))(
        attention_output
    )
    attention_output = layers.LayerNormalization(
        epsilon=1e-6, name="encoder_{}_att_layernormalization".format(i)
    )(query + attention_output)

    # Feed-forward layer
    ffn = keras.Sequential(
        [
            layers.Dense(config.FF_DIM, activation="relu"),
            layers.Dense(config.EMBED_DIM),
        ],
        name="encoder_{}_ffn".format(i),
    )
    ffn_output = ffn(attention_output)
    ffn_output = layers.Dropout(0.1, name="encoder_{}_ffn_dropout".format(i))(
        ffn_output
    )
    sequence_output = layers.LayerNormalization(
        epsilon=1e-6, name="encoder_{}_ffn_layernormalization".format(i)
    )(attention_output + ffn_output)
    return sequence_output


loss_fn = keras.losses.SparseCategoricalCrossentropy(reduction=None)
loss_tracker = keras.metrics.Mean(name="loss")


class MaskedLanguageModel(keras.Model):

    def compute_loss(self, x=None, y=None, y_pred=None, sample_weight=None):

        loss = loss_fn(y, y_pred, sample_weight)
        loss_tracker.update_state(loss, sample_weight=sample_weight)
        return keras.ops.sum(loss)

    def compute_metrics(self, x, y, y_pred, sample_weight):

        # Return a dict mapping metric names to current value
        return {"loss": loss_tracker.result()}

    @property
    def metrics(self):
        # We list our `Metric` objects here so that `reset_states()` can be
        # called automatically at the start of each epoch
        # or at the start of `evaluate()`.
        # If you don't implement this property, you have to call
        # `reset_states()` yourself at the time of your choosing.
        return [loss_tracker]


def create_masked_language_bert_model():
    inputs = layers.Input(shape=(config.MAX_LEN,), dtype="int64", name="")

    word_embeddings = layers.Embedding(
        config.VOCAB_SIZE, config.EMBED_DIM, name="word_embedding"
    )(inputs)
    position_embeddings = keras_hub.layers.PositionEmbedding(
        sequence_length=config.MAX_LEN,
        name=""
    )(word_embeddings)
    embeddings = word_embeddings + position_embeddings

    encoder_output = embeddings
    for i in range(1, config.NUM_LAYERS+1):
        encoder_output = bert_module(encoder_output, encoder_output, encoder_output, i)

    mlm_output = layers.Dense(config.VOCAB_SIZE, name="mlm_cls", activation="softmax")(
        encoder_output
    )
    mlm_model = MaskedLanguageModel(inputs, mlm_output, name="masked_bert_model")

    optimizer = keras.optimizers.Adam(learning_rate=config.LR)
    mlm_model.compile(optimizer=optimizer)
    return mlm_model


id2token = dict(enumerate(vectorize_layer.get_vocabulary()))
token2id = {y: x for x, y in id2token.items()}


class MaskedTextGenerator(keras.callbacks.Callback):
    def __init__(self, sample_tokens, top_k=5):
        self.sample_tokens = sample_tokens
        self.k = top_k

    def decode(self, tokens):
        return " ".join([id2token[t] for t in tokens if t != 0])

    def convert_ids_to_tokens(self, id):
        return id2token[id]

    def on_epoch_end(self, epoch, logs=None):
        prediction = self.model.predict(self.sample_tokens)

        masked_index = np.where(self.sample_tokens == mask_token_id)
        masked_index = masked_index[1]
        mask_prediction = prediction[0][masked_index]

        top_indices = mask_prediction[0].argsort()[-self.k :][::-1]
        values = mask_prediction[0][top_indices]

        for i in range(len(top_indices)):
            p = top_indices[i]
            v = values[i]
            tokens = np.copy(sample_tokens[0])
            tokens[masked_index[0]] = p
            result = {
                "input_text": self.decode(sample_tokens[0].numpy()),
                "prediction": self.decode(tokens),
                "probability": v,
                "predicted mask token": self.convert_ids_to_tokens(p),
            }
            pprint(result)


sample_tokens = vectorize_layer(["I have watched this [mask] and it was awesome"])

bert_masked_model = create_masked_language_bert_model()
bert_masked_model.summary()

Model: "masked_bert_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_embedding      │ (None, 128, 128)  │  3,840,000 │ input_layer_6[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ position_embedding… │ (None, 128, 128)  │     16,384 │ word_embedding[0… │
│ (PositionEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (None, 128, 128)  │          0 │ word_embedding[0… │
│                     │                   │            │ position_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_multihea… │ (None, 128, 128)  │     66,048 │ add_10[0][0],     │
│ (MultiHeadAttentio… │                   │            │ add_10[0][0],     │
│                     │                   │            │ add_10[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_att_drop… │ (None, 128, 128)  │          0 │ encoder_1_multih… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_11 (Add)        │ (None, 128, 128)  │          0 │ add_10[0][0],     │
│                     │                   │            │ encoder_1_att_dr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_att_laye… │ (None, 128, 128)  │        256 │ add_11[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_ffn       │ (None, 128, 128)  │    131,712 │ encoder_1_att_la… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_ffn_drop… │ (None, 128, 128)  │          0 │ encoder_1_ffn[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (None, 128, 128)  │          0 │ encoder_1_att_la… │
│                     │                   │            │ encoder_1_ffn_dr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1_ffn_laye… │ (None, 128, 128)  │        256 │ add_12[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_2_multihea… │ (None, 128, 128)  │     66,048 │ encoder_1_ffn_la… │
│ (MultiHeadAttentio… │                   │            │ encoder_1_ffn_la… │
│                     │                   │            │ encoder_1_ffn_la… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_2_att_drop… │ (None, 128, 128)  │          0 │ encoder_2_multih… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_13 (Add)        │ (None, 128, 128)  │          0 │ encoder_1_ffn_la… │
│                     │                   │            │ encoder_2_att_dr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_2_att_laye… │ (None, 128, 128)  │        256 │ add_13[0][0]      │
│ (LayerNormalizatio… │                   │            │                 

 Total params: 8,122,928 (30.99 MB)

 Trainable params: 8,122,928 (30.99 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
class MaskedTextGenerator(keras.callbacks.Callback):
    def __init__(self, sample_tokens, top_k=5):
        self.sample_tokens = sample_tokens
        self.k = top_k

    def decode(self, tokens):
        return " ".join([id2token[t] for t in tokens if t != 0])

    def convert_ids_to_tokens(self, id):
        return id2token[id]

    def on_epoch_end(self, epoch, logs=None):
        prediction = self.model.predict(self.sample_tokens)

        masked_index = np.where(self.sample_tokens == mask_token_id)
        masked_index = masked_index[1]
        mask_prediction = prediction[0][masked_index]

        top_indices = mask_prediction[0].argsort()[-self.k :][::-1]
        values = mask_prediction[0][top_indices]

        for i in range(len(top_indices)):
            p = top_indices[i]
            v = values[i]
            tokens = np.copy(sample_tokens[0])
            tokens[masked_index[0]] = p
            result = {
                "input_text": self.decode(sample_tokens[0].numpy()),
                "prediction": self.decode(tokens),
                "probability": v,
                "predicted mask token": self.convert_ids_to_tokens(p),
            }
            pprint(result)
generator_callback = MaskedTextGenerator(sample_tokens.numpy())

earlyStopping_callback = EarlyStopping(
    monitor="loss",
    min_delta=0.005,
    patience=5,
    verbose=0,
    mode="auto",
    baseline=None,
    restore_best_weights=True,
)

checkpoint_callback = ModelCheckpoint(
    os.path.join("models", "model_weights_cp_best.weights.h5"), 
    save_best_only=True,
    monitor="loss",
    save_weights_only=True,
    mode="min",
    verbose=1
)

In [17]:
bert_masked_model.fit(mlm_ds_small, epochs=50, callbacks=[generator_callback, checkpoint_callback, earlyStopping_callback])
save_model_weights(bert_masked_model, "bert_masked_model_v2.weights.h5", "models")

Epoch 1/50


ValueError: Input 0 of layer "masked_bert_model" is incompatible with the layer: expected shape=(None, 128), found shape=(None, 256)